# Eval Original PowerPaint Model
Chi danh gia model goc (JunhaoZhuang/PowerPaint_v2) tren COCO val

In [1]:
from google.colab import drive
from pathlib import Path
import os, sys, json, yaml, shutil, subprocess

drive.mount('/content/gdrive')

DRIVE_WORKSPACE = Path('/content/gdrive/MyDrive/Eval_Original')
REPO_DIR = DRIVE_WORKSPACE / 'PowerPaint'
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))

print('Python =', sys.version)
print('REPO_DIR =', REPO_DIR)
print('WORKSPACE =', DRIVE_WORKSPACE)

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).
Python = 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
REPO_DIR = /content/gdrive/MyDrive/Eval_Original/PowerPaint
WORKSPACE = /content/gdrive/MyDrive/Eval_Original


In [2]:
# Config
base_model = str(REPO_DIR / 'checkpoints' / 'ppt-v2' / 'realisticVisionV60B1_v51VAE')
powerpaint_ckpt = str(REPO_DIR / 'checkpoints' / 'ppt-v2' / 'PowerPaint_Brushnet')
results_root = DRIVE_WORKSPACE / 'results' / 'original'
pp_steps = 45
pp_guidance_removal = 10.0
pp_guidance_inpaint = 7.5
pp_input_mode_removal = 'original'  # 'masked' hoac 'original'

CLEAN_ZIP_PATH = "/content/gdrive/MyDrive/COCO_Raw_Data/coco_full_clean.zip"
BASE_DATA_DIR = "/content/coco_full_clean"
SPLIT = "val"
DATASET_DIR = os.path.join(BASE_DATA_DIR, SPLIT)


In [3]:
# Cai cac goi can thiet cho Colab theo bo version on dinh hon
import sys
import subprocess
import importlib.metadata as importlib_metadata
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Hay chuyen Colab sang GPU truoc khi chay notebook nay.')

py312_plus = sys.version_info >= (3, 12)
transformers_version = '4.39.3' if py312_plus else '4.28.0'
tokenizers_package = ['tokenizers==0.15.2'] if py312_plus else []

required_versions = {
    'diffusers': '0.27.0',
    'transformers': transformers_version,
    'huggingface_hub': '0.20.2',
    'accelerate': '0.31.0',
    'controlnet-aux': '0.0.3',
    'pillow': '10.3.0',
    'torchmetrics': '0.11.4',
    'torch-fidelity': '0.3.0',
}

def installed_version(dist_name):
    try:
        return importlib_metadata.version(dist_name)
    except importlib_metadata.PackageNotFoundError:
        return None

core_packages = [
    'diffusers==0.27.0',
    f'transformers=={transformers_version}',
    'huggingface_hub==0.20.2',
    'accelerate==0.31.0',
    'controlnet-aux==0.0.3',
    'safetensors>=0.4.3',
    'pillow==10.3.0',
    *tokenizers_package,
]

extra_packages = [
    'mmengine', 'omegaconf', 'packaging', 'imageio', 'opencv-python-headless',
    'albumentations', 'pycocotools', 'torchmetrics==0.11.4', 'torch-fidelity==0.3.0',
    'tqdm', 'pandas', 'pyyaml', 'ftfy', 'scipy', 'sentencepiece', 'einops', 'timm',
]

print('Python =', sys.version)
print('Checking package versions...')
mismatches = {name: (installed_version(name), want) for name, want in required_versions.items() if installed_version(name) != want}

# peft moi san tren Colab co the xung dot voi accelerate 0.31.0 cua PowerPaint.
peft_version = installed_version('peft')
if peft_version is not None:
    print(f'Removing incompatible peft=={peft_version} ...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'uninstall', '-y', 'peft'])

if mismatches:
    print('Installing/upgrading mismatched packages only:')
    for name, (have, want) in mismatches.items():
        print(f'  - {name}: {have} -> {want}')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--upgrade', '--no-cache-dir', *core_packages])
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--upgrade', '--no-cache-dir', *extra_packages])
else:
    print('Required package versions already installed.')

print('Install step complete. Neu ban dang o runtime moi, co the chay tiep ma khong can restart.')
print('Neu truoc do da import PIL/torchvision/torchmetrics trong runtime nay, hay Runtime -> Restart session roi chay lai tu dau.')


Python = 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Checking package versions...
Required package versions already installed.
Install step complete. Neu ban dang o runtime moi, co the chay tiep ma khong can restart.
Neu truoc do da import PIL/torchvision/torchmetrics trong runtime nay, hay Runtime -> Restart session roi chay lai tu dau.


In [4]:
import numpy as np
import torch
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
from safetensors.torch import load_model
from transformers import CLIPTextModel
from powerpaint.models.BrushNet_CA import BrushNetModel
from powerpaint.models.unet_2d_condition import UNet2DConditionModel
from powerpaint.pipelines.pipeline_PowerPaint_Brushnet_CA import StableDiffusionPowerPaintBrushNetPipeline
from powerpaint.utils.utils import TokenizerWrapper, add_tokens, EmbeddingLayerWithFixes

print('Torch:', torch.__version__, 'CUDA:', torch.cuda.is_available())
print('Transformers:', __import__('transformers').__version__)
print('Diffusers:', __import__('diffusers').__version__)
print('Pillow:', __import__('PIL').__version__)


Torch: 2.11.0+cu128 CUDA: True
Transformers: 4.39.3
Diffusers: 0.27.0
Pillow: 10.3.0


In [5]:
# Load original model
print("Loading original PowerPaint model...")

unet = UNet2DConditionModel.from_pretrained('runwayml/stable-diffusion-v1-5', subfolder='unet', torch_dtype=torch.float16)
text_encoder_brushnet = CLIPTextModel.from_pretrained('runwayml/stable-diffusion-v1-5', subfolder='text_encoder', torch_dtype=torch.float16)
brushnet = BrushNetModel.from_unet(unet).to('cuda', dtype=torch.float16)

tokenizer = TokenizerWrapper(from_pretrained=base_model, subfolder='tokenizer')
raw_tokenizer = tokenizer.wrapped
add_tokens(
    tokenizer=tokenizer,
    text_encoder=text_encoder_brushnet,
    placeholder_tokens=["P_ctxt", "P_obj"],
    initialize_tokens=["a", "a"],
    num_vectors_per_token=10,
)

load_model(brushnet, os.path.join(powerpaint_ckpt, 'diffusion_pytorch_model.safetensors'))
text_encoder_brushnet.load_state_dict(
    torch.load(os.path.join(powerpaint_ckpt, 'pytorch_model.bin')), strict=False
)

pipe = StableDiffusionPowerPaintBrushNetPipeline.from_pretrained(
    base_model,
    brushnet=brushnet,
    unet=UNet2DConditionModel.from_pretrained(base_model, subfolder='unet', torch_dtype=torch.float16),
    text_encoder_brushnet=text_encoder_brushnet,
    tokenizer=raw_tokenizer,
    safety_checker=None,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=False,
)

# promptU / negative_promptU di qua pipe.text_encoder, nen can dong bo external embeddings
main_embedding_layer = pipe.text_encoder.text_model.embeddings.token_embedding
if not isinstance(main_embedding_layer, EmbeddingLayerWithFixes):
    pipe.text_encoder.text_model.embeddings.token_embedding = EmbeddingLayerWithFixes(main_embedding_layer)
    main_embedding_layer = pipe.text_encoder.text_model.embeddings.token_embedding

copied_embeddings = []
for emb in text_encoder_brushnet.text_model.embeddings.token_embedding.external_embeddings:
    copied = {k: v for k, v in emb.items() if k not in {'embedding'}}
    copied['embedding'] = emb['embedding'].detach().clone()
    copied['trainable'] = False
    copied_embeddings.append(copied)

if copied_embeddings:
    main_embedding_layer.add_embeddings(copied_embeddings)

pipe.tokenizer = tokenizer
pipe = pipe.to('cuda')
pipe.set_progress_bar_config(disable=True)
print("Model loaded")


Loading original PowerPaint model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_token.py:88: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


06/29 10:38:39 - mmengine - INFO - Successfully add external embeddings: P_ctxt, P_obj.
06/29 10:38:39 - mmengine - INFO - Successfully add trainable external embeddings: P_ctxt, P_obj


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/transformers/models/clip/feature_extraction_clip.py:28: FutureWarning: The class CLIPFeatureExtractor is deprecated and will be removed in version 5 of Transformers. Please use CLIPImageProcessor instead.
  warnings.warn(


06/29 10:38:47 - mmengine - INFO - Successfully add external embeddings: P_ctxt, P_obj.
Model loaded


In [6]:
# Helper functions
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def _list_images(folder):
    folder = Path(folder)
    return sorted([p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS])

def _load_image(path, size=None):
    image = Image.open(path).convert("RGB")
    if size is not None:
        image = image.resize((size, size), Image.Resampling.BICUBIC)
    array = np.asarray(image, dtype=np.float32) / 255.0
    return torch.from_numpy(array).permute(2, 0, 1)

def _load_batch(paths, size, device):
    return torch.stack([_load_image(p, size=size) for p in paths], dim=0).to(device)

def _round_to_multiple_of_8(x):
    return max(8, (x // 8) * 8)

def _prepare_inference_inputs(image, mask, max_side=640, threshold=127):
    image = image.convert('RGB')
    mask = mask.convert('L').resize(image.size, Image.NEAREST)
    w, h = image.size
    scale = min(1.0, max_side / max(w, h))
    resized_w = _round_to_multiple_of_8(int(round(w * scale)))
    resized_h = _round_to_multiple_of_8(int(round(h * scale)))
    image = image.resize((resized_w, resized_h), Image.Resampling.LANCZOS)
    mask_gray = mask.resize((resized_w, resized_h), Image.NEAREST)
    binary_mask = mask_gray.point(lambda p: 255 if p > threshold else 0, mode='L')
    masked_image = Image.composite(Image.new('RGB', image.size, (0, 0, 0)), image, binary_mask)
    return image, binary_mask, masked_image

def _compose_inpainted_result(base_image, generated, mask):
    generated = generated.convert('RGB').resize(base_image.size, Image.Resampling.LANCZOS)
    return Image.composite(generated, base_image, mask.convert('L'))

def _save_debug_views(debug_dir, stem, original, mask, pipe_input, generated, result):
    from PIL import ImageDraw, ImageFont

    labels = ['original', 'mask', 'pipe_input', 'generated', 'final']
    panels = [
        original.convert('RGB'),
        mask.convert('RGB'),
        pipe_input.convert('RGB'),
        generated.convert('RGB').resize(original.size, Image.Resampling.LANCZOS),
        result.convert('RGB'),
    ]
    label_height = 32
    try:
        font = ImageFont.truetype('DejaVuSans.ttf', 18)
    except OSError:
        font = ImageFont.load_default()
    total_width = sum(panel.width for panel in panels)
    max_height = max(panel.height for panel in panels) + label_height
    canvas = Image.new('RGB', (total_width, max_height), (0, 0, 0))
    draw = ImageDraw.Draw(canvas)
    offset_x = 0
    for label, panel in zip(labels, panels):
        canvas.paste(panel, (offset_x, label_height))
        bbox = draw.textbbox((0, 0), label, font=font)
        text_w = bbox[2] - bbox[0]
        text_h = bbox[3] - bbox[1]
        text_x = offset_x + max(0, (panel.width - text_w) // 2)
        text_y = max(0, (label_height - text_h) // 2 - 1)
        draw.text((text_x, text_y), label, fill=(255, 255, 255), font=font)
        offset_x += panel.width
    out_path = Path(debug_dir) / f'{stem}_debug.jpg'
    canvas.save(out_path, quality=95)
    return out_path

def evaluate_pairs(pairs, prompts, batch_size=8, image_size=299, device=None, clip_model_name="openai/clip-vit-base-patch16"):
    from torchmetrics.image.fid import FrechetInceptionDistance
    from torchmetrics.image.lpip import LearnedPerceptualImagePatchSimilarity
    from torchmetrics.multimodal import CLIPScore

    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
    device = torch.device(device)

    real_paths = [x[0] for x in pairs]
    gen_paths = [x[1] for x in pairs]

    fid = FrechetInceptionDistance(feature=2048, normalize=True).to(device)
    for i in range(0, len(real_paths), batch_size):
        fid.update(_load_batch(real_paths[i:i+batch_size], image_size, device), real=True)
    for i in range(0, len(gen_paths), batch_size):
        fid.update(_load_batch(gen_paths[i:i+batch_size], image_size, device), real=False)
    fid_value = float(fid.compute().detach().cpu().item())

    lpips = LearnedPerceptualImagePatchSimilarity(net_type="alex", normalize=True).to(device)
    for i in range(0, len(pairs), batch_size):
        chunk = pairs[i:i+batch_size]
        rb = _load_batch([x[0] for x in chunk], image_size, device)
        gb = _load_batch([x[1] for x in chunk], image_size, device)
        lpips.update(gb, rb)
    lpips_value = float(lpips.compute().detach().cpu().item())

    clip = CLIPScore(model_name_or_path=clip_model_name).to(device)
    for i in range(0, len(gen_paths), batch_size):
        clip.update(_load_batch(gen_paths[i:i+batch_size], image_size, device), list(prompts[i:i+batch_size]))
    clip_value = float(clip.compute().detach().cpu().item())

    return {"fid": fid_value, "lpips": lpips_value, "clip_score": clip_value, "num_samples": len(pairs)}


In [7]:
# Unzip dataset
if not os.path.exists(os.path.join(DATASET_DIR, "metadata.json")):
    print("Extracting clean dataset...")
    if os.path.exists(BASE_DATA_DIR):
        shutil.rmtree(BASE_DATA_DIR)
    !cp "{CLEAN_ZIP_PATH}" /content/temp_data.zip
    !unzip -q /content/temp_data.zip -d /content/
    !rm /content/temp_data.zip
    nested_dir = "/content/content/coco_full_clean"
    if os.path.exists(nested_dir):
        os.makedirs(BASE_DATA_DIR, exist_ok=True)
        !mv "{nested_dir}"/* "{BASE_DATA_DIR}"/
        !rm -rf /content/content

with open(os.path.join(DATASET_DIR, "metadata.json"), "r") as f:
    full_metadata = json.load(f)

metadata_run = []
for item in full_metadata:
    metadata_run.append({
        "image_path": os.path.join(BASE_DATA_DIR, item["image_path"]),
        "mask_path": os.path.join(BASE_DATA_DIR, item["mask_path"]),
        "caption": item.get("caption", "")
    })

METADATA_RUN_PATH = os.path.join(DATASET_DIR, "metadata_run.json")
with open(METADATA_RUN_PATH, "w") as f:
    json.dump(metadata_run, f, indent=2)

print(f"Samples: {len(metadata_run)}, metadata: {METADATA_RUN_PATH}")

Samples: 2477, metadata: /content/coco_full_clean/val/metadata_run.json


In [8]:
# Inference: object_removal
with open(METADATA_RUN_PATH) as f:
    items = json.load(f)

out_dir = str(results_root / 'object_removal')
debug_dir = str(results_root / 'object_removal_debug')
Path(out_dir).mkdir(parents=True, exist_ok=True)
Path(debug_dir).mkdir(parents=True, exist_ok=True)

for item in tqdm(items, desc='removal'):
    img = Image.open(item['image_path']).convert('RGB')
    msk = Image.open(item['mask_path']).convert('L')
    image, mask, masked_img = _prepare_inference_inputs(img, msk)

    if pp_input_mode_removal == 'original':
        pipe_input = image
    elif pp_input_mode_removal == 'masked':
        pipe_input = masked_img
    else:
        raise ValueError(f'pp_input_mode_removal khong hop le: {pp_input_mode_removal}')

    out = pipe(
        promptA='P_ctxt', promptB='P_ctxt', promptU='P_ctxt',
        negative_promptA='P_obj', negative_promptB='P_obj', negative_promptU='',
        tradoff=1.0, image=pipe_input, mask=mask,
        num_inference_steps=pp_steps, guidance_scale=pp_guidance_removal,
        generator=torch.Generator(device='cuda').manual_seed(42),
        width=image.size[0], height=image.size[1],
    ).images[0]
    final = _compose_inpainted_result(image, out, mask)
    final.save(Path(out_dir) / Path(item['image_path']).name)
    stem = Path(item['image_path']).stem
    _save_debug_views(debug_dir, stem, image, mask, pipe_input, out, final)

print(f'Saved results to {out_dir}')
print(f'Saved debug views to {debug_dir}')


removal:   0%|          | 0/2477 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/diffusers/models/resnet.py:328: FutureWarning: `scale` is deprecated and will be removed in version 1.0.0. The `scale` argument is deprecated and will be ignored. Please remove it, as passing it will raise an error in the future. `scale` should directly be passed while calling the underlying pipeline component i.e., via `cross_attention_kwargs`.
  deprecate("scale", "1.0.0", deprecation_message)
/usr/local/lib/python3.12/dist-packages/diffusers/models/downsampling.py:136: FutureWarning: `scale` is deprecated and will be removed in version 1.0.0. The `scale` argument is deprecated and will be ignored. Please remove it, as passing it will raise an error in the future. `scale` should directly be passed while calling the underlying pipeline component i.e., via `cross_attention_kwargs`.
  deprecate("scale", "1.0.0", deprecation_message)
/usr/local/lib/python3.12/dist-packages/diffusers/models/upsampling.py:147: FutureWarning: `scale` is deprecated and

KeyboardInterrupt: 

In [ ]:
# Eval metrics
gen_dir = Path(str(results_root / 'object_removal'))
gen_map = {p.name: p for p in _list_images(gen_dir)}

pairs, prompts = [], []
with open(METADATA_RUN_PATH) as f:
    items = json.load(f)
for it in items:
    real_path = Path(it['image_path'])
    name = real_path.name
    if name in gen_map:
        pairs.append((real_path, gen_map[name]))
        prompts.append('')

print(f'Matched: {len(pairs)}')
result = evaluate_pairs(pairs, prompts, batch_size=8)
print(pd.DataFrame([{**result}]))